# Imports & Setup

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

DATA_DIR = '/kaggle/input/competitions/santander-customer-transaction-prediction'
INPUT_ARTIFACTS_DIR = '/kaggle/input/notebooks/tahaarif23/santander-customer-transaction-prediction/artifacts' 

WORKDIR = '/kaggle/working'
OUTPUT_ARTIFACTS_DIR = os.path.join(WORKDIR, 'final_report_artifacts')
os.makedirs(OUTPUT_ARTIFACTS_DIR, exist_ok=True)

# 1. GENERATE EDA PLOTS

In [2]:
print('1) Loading data for EDA artifacts...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# EDA 1: Target imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x='target', hue='target', palette='viridis', legend=False)
plt.title('Santander Dataset: Target Class Imbalance (90/10)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ARTIFACTS_DIR, 'eda_target_imbalance.png'))
plt.close()

1) Loading data for EDA artifacts...


# EDA 2: Representative feature distribution by target

In [3]:
feature_for_plot = 'var_81' if 'var_81' in train_df.columns else 'var_0'
plt.figure(figsize=(8, 5))
sns.histplot(data=train_df, x=feature_for_plot, hue='target', bins=50, kde=True, palette='seismic')
plt.title(f'Distribution of Top Feature ({feature_for_plot}) by Target Class')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ARTIFACTS_DIR, f'eda_{feature_for_plot}_distribution.png'))
plt.close()

# 2. LOAD SAVED ARTIFACTS 


In [4]:
print('2) Loading model artifacts for local SHAP plots...')
model_path = os.path.join(INPUT_ARTIFACTS_DIR, 'champion_model_full.pkl')
features_path = os.path.join(INPUT_ARTIFACTS_DIR, 'selected_features.pkl')
scaler_path = os.path.join(INPUT_ARTIFACTS_DIR, 'scaler.joblib')

if not (os.path.exists(model_path) and os.path.exists(features_path) and os.path.exists(scaler_path)):
    raise FileNotFoundError(f"Required artifacts not found in {INPUT_ARTIFACTS_DIR}. Check your input path!")

explainer_model = joblib.load(model_path)
selected_features = joblib.load(features_path)
scaler = joblib.load(scaler_path)

X_full = train_df.drop(columns=['target', 'ID_code'])
feature_names_full = X_full.columns.tolist()
selected_indices = [feature_names_full.index(f) for f in selected_features]

2) Loading model artifacts for local SHAP plots...


# 3. GENERATE SHAP WATERFALL PLOTS 

In [5]:
print('3) Generating 3 local SHAP waterfall plots...')

# Use three positive-class examples for local explanations
positive_idx = train_df.index[train_df['target'] == 1]
if len(positive_idx) < 3:
    sample_indices = train_df.sample(3, random_state=42).index
else:
    sample_indices = train_df.loc[positive_idx].sample(3, random_state=42).index

# Transform and convert to DataFrame so SHAP uses real feature names (e.g., var_190)
X_sample_scaled = scaler.transform(X_full.loc[sample_indices])[:, selected_indices]
X_sample_df = pd.DataFrame(X_sample_scaled, columns=selected_features)

explainer = shap.TreeExplainer(explainer_model)
shap_values = explainer(X_sample_df)

for i in range(3):
    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f'Local Transaction Explanation (Positive Sample #{i+1})', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ARTIFACTS_DIR, f'shap_local_waterfall_{i+1}.png'), dpi=300)
    plt.close()

3) Generating 3 local SHAP waterfall plots...


# 4. EXPORT REPRODUCIBILITY JSON 

In [6]:
print('4) Writing reproducibility JSON...')
reproducibility_data = {
    'competition': 'Santander Customer Transaction Prediction',
    'framework': 'LightGBM',
    'validation_strategy': '5-Fold Stratified CV',
    'final_features_count': 180,
    'optimal_threshold': 0.6744,
    'metrics': {
        'OOF_ROC_AUC': 0.89483,
        'Public_Leaderboard': 0.89693,
        'Private_Leaderboard': 0.89412
    },
    'artifacts_dir': OUTPUT_ARTIFACTS_DIR
}

with open(os.path.join(OUTPUT_ARTIFACTS_DIR, 'reproducibility.json'), 'w', encoding='utf-8') as f:
    json.dump(reproducibility_data, f, indent=2)

print(f" Done! All final artifacts generated and saved to: {OUTPUT_ARTIFACTS_DIR}")

4) Writing reproducibility JSON...
 Done! All final artifacts generated and saved to: /kaggle/working/final_report_artifacts
